# 🤖 Notebook 07: Building a GPT from Scratch

We assemble a complete GPT model in MLX: token embedding + positional encoding + N transformer blocks + output projection. Then we train it on text and generate.

**Prerequisites:** Notebooks 01-06

**What you'll learn:**
1. Build a complete GPT model from transformer blocks\n2. Implement a training loop with cosine LR schedule\n3. Train on Shakespeare text and watch quality improve\n4. Generate text and see the model learn language patterns

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## ✅ Environment Validation

### 🧠 The Big Picture

Building a GPT model is like assembling a car from parts — the embedding layer is the fuel intake (converting raw text into usable form), the transformer blocks are the engine (doing the heavy processing), and the output head is the steering wheel (deciding what comes next). Training is like teaching someone to drive by showing them millions of examples.

### 💡 Why Does This Matter?

Why cosine learning rate schedule? It starts with a high learning rate for fast initial progress, then gradually reduces it for fine-grained optimization — like taking big steps when you're far from the destination and small steps when you're close. Why gradient clipping? It prevents catastrophically large updates that can destabilize training.

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## GPT Architecture Overview

```
Token IDs (batch, seq_len)
  │
  ▼
Token Embedding (vocab_size, d_model)
  │
  ▼
Positional Encoding (sinusoidal or RoPE)
  │
  ▼
N × TransformerBlock
  │
  ▼
Final RMSNorm
  │
  ▼
Output Projection (d_model → vocab_size)
  │
  ▼
Logits (batch, seq_len, vocab_size)
```

## Complete GPTModel Implementation

This is the full model with `__call__` for training and `generate` for text generation.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import math
import time
import numpy as np
import matplotlib.pyplot as plt

class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
    def __call__(self, x):
        return self.w2(nn.silu(self.w_gate(x)) * self.w1(x))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    def __call__(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        K = self.W_k(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        V = self.W_v(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float('-inf')), scores)  # shape: see output
        weights = mx.softmax(scores, axis=-1)  # shape: see output
        out = (weights @ V).transpose(0, 2, 1, 3).reshape(B, T, D)
        return self.W_o(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn_norm = nn.RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)
    def __call__(self, x, mask=None):
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

class GPTModel(nn.Module):
    """Complete GPT model."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)  # learned positional
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.final_norm = nn.RMSNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.max_seq_len = max_seq_len
    
    def __call__(self, token_ids):
        B, T = token_ids.shape
        tok_emb = self.token_emb(token_ids)  # (B, T, d_model)
        pos = mx.arange(T)  # shape: see output
        pos_emb = self.pos_emb(pos)  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        mask = mx.tril(mx.ones((T, T)))  # causal mask  # shape: see output
        for block in self.blocks:
            x = block(x, mask)  # shape: function output
        x = self.final_norm(x)  # (B, T, d_model)
        return self.output_proj(x)  # (B, T, vocab_size)
    
    def generate(self, prompt_ids, max_tokens=50, temperature=1.0):
        tokens = list(prompt_ids.tolist()) if hasattr(prompt_ids, 'tolist') else list(prompt_ids)
        for _ in range(max_tokens):
            ctx = tokens[-self.max_seq_len:]
            x = mx.array([ctx])
            logits = self(x)  # (1, T, vocab_size)
            logits = logits[0, -1, :] / temperature
            probs = mx.softmax(logits, axis=-1)
            mx.eval(probs)
            probs_np = np.array(probs, dtype=np.float64)
            probs_np /= probs_np.sum()  # re-normalize: float32→float64 can break np.random.choice
            next_token = int(np.random.choice(len(probs_np), p=probs_np))
            tokens.append(next_token)
        return tokens

print('GPTModel defined ✅')

### 🔍 What Just Happened?

We just trained a language model from scratch! The model learned to predict the next character by processing sequences through embedding → transformer blocks → output projection. The training loop pattern (forward → loss → backward → clip → update → eval) is the same pattern used to train GPT-4, just at a much smaller scale.

## Model Inspection

Let's count parameters and estimate memory for different configs.

In [ ]:
configs = {
    'tiny':   dict(vocab_size=256, d_model=64,  n_heads=4, n_layers=2, d_ff=256,  max_seq_len=128),
    'small':  dict(vocab_size=256, d_model=128, n_heads=4, n_layers=4, d_ff=512,  max_seq_len=256),
    'medium': dict(vocab_size=256, d_model=256, n_heads=8, n_layers=6, d_ff=1024, max_seq_len=256),
}

for name, cfg in configs.items():
    model = GPTModel(**cfg)
    # Simple param count
    n_params = sum(p.size for _, p in mx.utils.tree_flatten(model.parameters()))
    mem_mb = n_params * 4 / 1e6  # float32
    print(f'{name:>7s}: {n_params:>10,} params ({mem_mb:.1f} MB float32)')

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

## Data Preparation

We'll use a simple text dataset with character-level tokenization for training our tiny GPT.

In [ ]:
# Download tiny shakespeare or use inline text
text = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die, to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil."""

# Character-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = mx.array(encode(text))
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'Vocab size: {vocab_size}')
print(f'Train tokens: {len(train_data)}, Val tokens: {len(val_data)}')
print(f'Sample: "{text[:60]}..."')
print(f'Encoded: {encode(text[:20])}')

## Training Loop

Standard MLX training: `nn.value_and_grad` → `optimizer.update` → `mx.eval`.

In [ ]:
# Config
seq_len = 64
batch_size = 8
lr = 3e-4
n_steps = 300

model = GPTModel(vocab_size=vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=seq_len)
optimizer = optim.AdamW(learning_rate=lr, weight_decay=0.1)

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = mx.random.randint(0, len(d) - seq_len, shape=(batch_size,))
    mx.eval(ix)
    ix = ix.tolist()
    x = mx.stack([d[i:i+seq_len] for i in ix])
    y = mx.stack([d[i+1:i+seq_len+1] for i in ix])
    return x, y

def loss_fn(model, x, y):
    logits = model(x)  # (B, T, vocab_size)
    return nn.losses.cross_entropy(logits, y, reduction='mean')

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

losses = []
t0 = time.perf_counter()
for step in range(n_steps):
    x, y = get_batch('train')
    loss, grads = loss_and_grad_fn(model, x, y)
    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state, loss)
    loss_val = loss.item()
    losses.append(loss_val)
    if step % 50 == 0 or step == n_steps - 1:
        print(f'Step {step:4d} | Loss: {loss_val:.4f} | PPL: {math.exp(loss_val):.1f}')

t1 = time.perf_counter()
print(f'\nTraining done in {t1-t0:.1f}s')

## Loss Curve & Text Generation

In [ ]:
from utils.viz import plot_loss_curve
fig = plot_loss_curve(losses, title='GPT Training Loss')
plt.show()

# Generate text
prompt = 'To be'
prompt_ids = encode(prompt)
generated = model.generate(prompt_ids, max_tokens=100, temperature=0.8)
print(f'Prompt: "{prompt}"')
print(f'Generated: "{decode(generated)}"')

## 🔧 Deep Rework: Full Training Pipeline

The sections below upgrade this notebook from a toy demo to a proper training pipeline with:
- 📦 Real dataset (tiny_shakespeare) with train/val split
- 📈 Cosine LR schedule with linear warmup
- ✂️ Gradient clipping
- 💾 Checkpoint save/load with NaN recovery
- 📊 Evaluation loop (validation loss + perplexity)
- ✍️ Generation samples showing quality progression

All heavy lifting lives in `utils/training.py` — notebook cells just import and call.

## 📦 Task 9.1 — Dataset Download & Preparation

💡 We use the **tiny_shakespeare** dataset (~1 MB) — all of Shakespeare's works concatenated. The `download_tiny_shakespeare` function fetches it once and caches locally, then splits 90/10 into train/val.

🎯 **Interview tip:** Real training always needs a held-out validation set to detect overfitting.

In [ ]:
from utils.training import download_tiny_shakespeare, CharTokenizer

# Download and split dataset
train_text, val_text = download_tiny_shakespeare(data_dir="data", val_fraction=0.1)
print(f"Train chars: {len(train_text):,}  |  Val chars: {len(val_text):,}")
print(f"Sample: {train_text[:120]}…")

# Build character tokenizer
tokenizer = CharTokenizer(train_text + val_text)
print(f"\n💡 Vocab size: {tokenizer.vocab_size}")
print(f"Encode 'Hello': {tokenizer.encode('Hello')}")
print(f"Decode back:    {tokenizer.decode(tokenizer.encode('Hello'))}")

# Tokenize into MLX arrays
train_ids = mx.array(tokenizer.encode(train_text))
val_ids = mx.array(tokenizer.encode(val_text))
print(f"\nTrain tokens: {train_ids.shape[0]:,}  |  Val tokens: {val_ids.shape[0]:,}")

## 📈 Task 9.2 — Full Training Pipeline

⚡ **Cosine LR schedule with linear warmup** — the standard recipe used by GPT-3, LLaMA, and most modern LLMs:

$$\text{lr}(t) = \begin{cases} \text{max\_lr} \times \frac{t}{\text{warmup}} & t < \text{warmup} \\ \text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})\left(1 + \cos\frac{\pi \cdot t}{\text{max\_steps}}\right) & t \geq \text{warmup} \end{cases}$$

💡 **Why warmup?** Starting with a high learning rate is like trying to park a car at full speed — the model's randomly initialized weights produce wild gradients early on, and a large LR amplifies that chaos, often causing divergence or NaN losses. Warmup gradually increases the rate over the first few steps so the optimizer can "feel out" the loss landscape before taking big steps. Once the gradients stabilize, we ramp up to the full learning rate and then cosine-decay it back down.

⚠️ **Gradient clipping** prevents a single bad batch from blowing up training. We clip the global norm to `max_norm`.

In [ ]:
from utils.training import cosine_lr_schedule, clip_grad_norm

# --- Visualize the LR schedule ---
max_lr, min_lr = 3e-4, 1e-5
warmup_steps, max_steps = 20, 200
steps = list(range(max_steps))
lrs = [cosine_lr_schedule(s, warmup_steps, max_steps, max_lr, min_lr) for s in steps]

plt.figure(figsize=(8, 3))
plt.plot(steps, lrs, linewidth=2)
plt.axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='warmup end')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.title('💡 Cosine LR Schedule with Linear Warmup')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()
print(f"LR at step 0: {lrs[0]:.2e}  |  peak: {max(lrs):.2e}  |  final: {lrs[-1]:.2e}")

The next cell continues the implementation:

**--- Full training loop with cosine LR + gradient clipping ---**

In [ ]:
# --- Full training loop with cosine LR + gradient clipping ---
seq_len = 64
batch_size = 8
max_lr = 3e-4
min_lr = 1e-5
warmup_steps = 20
n_steps = 200
clip_norm = 1.0

# Fresh model on the real dataset
model = GPTModel(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=2, d_ff=256, max_seq_len=seq_len,
)
optimizer = optim.AdamW(learning_rate=max_lr, weight_decay=0.1)

def get_batch(data, batch_size, seq_len):
    ix = np.random.randint(0, len(data) - seq_len, size=(batch_size,))
    x = mx.stack([data[i:i+seq_len] for i in ix])
    y = mx.stack([data[i+1:i+seq_len+1] for i in ix])
    return x, y

def loss_fn(model, x, y):
    logits = model(x)
    return nn.losses.cross_entropy(logits, y, reduction='mean')

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

train_losses, grad_norms, lr_history = [], [], []
t0 = time.perf_counter()

for step in range(n_steps):
    # Cosine LR schedule
    lr = cosine_lr_schedule(step, warmup_steps, n_steps, max_lr, min_lr)
    optimizer.learning_rate = lr
    lr_history.append(lr)

    x, y = get_batch(train_ids, batch_size, seq_len)
    loss, grads = loss_and_grad_fn(model, x, y)

    # Gradient clipping
    grads, gnorm = clip_grad_norm(grads, clip_norm)
    grad_norms.append(gnorm)

    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state, loss)

    loss_val = loss.item()
    train_losses.append(loss_val)

    if step % 50 == 0 or step == n_steps - 1:
        print(f"Step {step:4d} | Loss: {loss_val:.4f} | PPL: {math.exp(loss_val):.1f} | "
              f"LR: {lr:.2e} | GradNorm: {gnorm:.3f}")

elapsed = time.perf_counter() - t0
print(f"\n⚡ Training done in {elapsed:.1f}s ({n_steps/elapsed:.0f} steps/s)")

## 💾 Task 9.4 — Checkpoint Save/Load & NaN Recovery

💡 `CheckpointManager` persists model weights, optimizer state, and step counter to disk. If training hits NaN, we reload the last good checkpoint and reduce LR.

⚠️ **Pitfall:** Without checkpointing, a NaN at step 900 of 1000 means starting over from scratch.

In [ ]:
from utils.training import CheckpointManager

ckpt_mgr = CheckpointManager(checkpoint_dir="checkpoints/gpt_demo")

# Save current training state
ckpt_path = ckpt_mgr.save(model, optimizer, step=n_steps)
print(f"💾 Saved checkpoint to: {ckpt_path}")

# Demonstrate round-trip: load into a fresh model
model_restored = GPTModel(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=2, d_ff=256, max_seq_len=seq_len,
)
optimizer_restored = optim.AdamW(learning_rate=max_lr, weight_decay=0.1)
restored_step = ckpt_mgr.load(model_restored, optimizer_restored, ckpt_path)
print(f"✅ Restored from step {restored_step}")

# Verify outputs match
test_x, _ = get_batch(val_ids, 2, seq_len)
out_orig = model(test_x)
out_rest = model_restored(test_x)
mx.eval(out_orig, out_rest)
max_diff = float(mx.max(mx.abs(out_orig - out_rest)).item())
print(f"🎯 Max output difference after restore: {max_diff:.2e} (should be ~0)")

# NaN detection demo
print(f"\n⚠️ NaN detection: {ckpt_mgr.detect_nan(float('nan'))} (True = would trigger recovery)")
print(f"   Normal loss:   {ckpt_mgr.detect_nan(2.5)} (False = all good)")

## 📊 Task 9.6 — Evaluation & Generation Samples

🎯 **Perplexity** = exp(cross-entropy loss). A perplexity of *P* means the model is as uncertain as choosing uniformly among *P* options at each step. Lower is better.

We show generation samples to visualize quality progression.

In [ ]:
from utils.training import evaluate, generate_text

# Evaluate on validation set
val_loss, ppl = evaluate(model, val_ids, batch_size=8, seq_len=seq_len, n_batches=20)
print(f"📊 Validation Loss: {val_loss:.4f}  |  Perplexity: {ppl:.1f}")

# Plot training loss + gradient norms
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(train_losses, alpha=0.7)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(grad_norms, alpha=0.7, color='orange')
axes[1].axhline(clip_norm, color='red', linestyle='--', alpha=0.5, label=f'clip={clip_norm}')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(lr_history, alpha=0.7, color='green')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate Schedule'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

The next cell continues the implementation:

**Generation samples — show quality progression**

In [ ]:
# Generation samples — show quality progression
prompts = ["ROMEO:", "To be", "The king"]
print("=" * 60)
print("✍️  Generation Samples (after training)")
print("=" * 60)
for prompt in prompts:
    generated = generate_text(model, tokenizer, prompt, max_tokens=150, temperature=0.8)
    print(f"\nPrompt: \"{prompt}\"")
    print(f"Generated: \"{generated[:200]}\"")
    print("-" * 60)

## 🧪 Try It Yourself

Experiment with your GPT model:

1. **Temperature**: Generate text with temperature=0.1, 0.5, 1.0, and 2.0. How does the output change? Why?

2. **Model size**: Try doubling d_model from 64 to 128. Does the loss decrease faster? How much more memory does it use?

3. **Training data**: Replace the training text with something different (a poem, code, etc.). How does the generated output change after training?

## 📜 History & Alternatives: The GPT Family

The GPT lineage traces the evolution of decoder-only transformers from research curiosity to the most capable AI systems:

| Year | Model | Team | Key Contribution |
|------|-------|------|-----------------|
| **2018** | **GPT-1** | OpenAI (Radford et al.) | Proved unsupervised pre-training + supervised fine-tuning works. 117M params, BooksCorpus. First to show a single pre-trained model transfers to many tasks. |
| **2019** | **GPT-2** | OpenAI (Radford et al.) | Scaled to 1.5B params, WebText dataset. Zero-shot task performance without fine-tuning. "Too dangerous to release" controversy sparked AI safety debate. |
| **2020** | **GPT-3** | OpenAI (Brown et al.) | 175B params, in-context learning via few-shot prompting. Showed scaling unlocks emergent abilities. Introduced the API-as-a-product model. |
| **2022** | **InstructGPT** | OpenAI (Ouyang et al.) | RLHF alignment — made GPT-3 follow instructions. Bridge from "predict next token" to "helpful assistant." |
| **2023** | **GPT-4** | OpenAI | Multimodal (text + vision), dramatically improved reasoning. Mixture-of-Experts architecture (rumored). Set new SOTA on professional exams. |
| **2024** | **GPT-4o** | OpenAI | Omni-modal: native audio, vision, and text in one model. Real-time voice conversation. |

### 🎯 Interview Insight: Why Decoder-Only Won

Early transformer work explored encoder-only (BERT), encoder-decoder (T5), and decoder-only (GPT) architectures. Decoder-only won for generation because:

1. **Simpler architecture** — one stack, one attention pattern (causal)
2. **Scales better** — all parameters contribute to generation
3. **Unified interface** — every task becomes "complete this text"
4. **KV-cache friendly** — causal attention enables efficient autoregressive decoding

### Alternatives & Parallel Tracks

- **BERT (2018)** — Encoder-only, bidirectional. Dominated NLU benchmarks but can't generate.
- **T5 (2019)** — Encoder-decoder, "text-to-text" framing. Still used for translation/summarization.
- **LLaMA (2023, Meta)** — Open-weight GPT-style models. Sparked the open-source LLM revolution.
- **Mistral/Mixtral (2023-24)** — Efficient architectures with sliding window attention and MoE.
- **Gemma (2024-25, Google)** — Compact, open models. Gemma 4 uses MoE + shared experts.

⚡ **The trend:** Each generation scales compute ~10-100×, but the core decoder-only transformer architecture from GPT-1 remains remarkably unchanged.

## Summary

We built a complete GPT model from scratch in MLX and trained it on Shakespeare with a production-grade pipeline:

- **Dataset**: tiny_shakespeare with 90/10 train/val split and character-level tokenization
- **LR Schedule**: Cosine decay with linear warmup (the GPT-3 recipe)
- **Gradient Clipping**: Global norm clipping to prevent training instability
- **Checkpointing**: Save/load model + optimizer state with NaN recovery
- **Evaluation**: Validation loss and perplexity tracking
- **Generation**: Temperature-controlled autoregressive sampling

**Next:** Notebook 08 — Training on Apple Silicon (memory profiling, mixed precision, gradient accumulation)